# GravityOCR in 10 minutes

Serve [`trillionlabs/GravityOCR`](https://huggingface.co/trillionlabs/GravityOCR) with self-speculative
decoding, read text, a table and a formula from image crops, and compare it with plain autoregressive
decoding of the same weights. Every step drives the files in this folder; [`README.md`](README.md) is
the reference for every setting.

The notebook runs on either of two backends, picked automatically (`BACKEND`, section 1). Both answer
the same OpenAI-compatible requests, so sections 2–6 are the same:

| `BACKEND` | machine | server | use it for |
|---|---|---|---|
| **`sglang`** | Linux with an **NVIDIA L40** or H100 | `serve.sh`: upstream SGLang + the GravityOCR patch | the tutorial environment and real serving |
| `mac` | Apple Silicon Mac | `serve_mac.py`: the MLX engine of the [Gravity](https://gravityocr.trillionlabs.co) app | trying it on a laptop, testing the notebook without an NVIDIA GPU |

| part | when | time |
|---|---|---|
| **Before you start**: build the server env (L40) or install Gravity (Mac) | once per machine | not counted (downloads several GB) |
| **Tutorial**: sections 1–6 | each run | 5–10 min; the first server start on an L40 compiles kernels |
| **Appendix**: what is inside, checkpoint check, security, troubleshooting | when you need it | — |

> **Run it with Claude Code.** Paste this prompt in a Claude Code session on the machine:
>
> *Read "Before you start" in `inference/gravityocr/getting_started.ipynb` and do the one-time setup
> for this machine (on Linux run `setup.sh` in the background). Then execute the notebook in place with
> `jupyter nbconvert --to notebook --execute --inplace` from a Python env that has `jupyter` and
> `pillow` (not the server venv), and summarize the PASS/FAIL lines, the self-spec vs causal table and
> the tutorial time.*
>
> Settings come from environment variables (section 1), so a headless run needs no edits, and every
> server the notebook starts is stopped when the kernel exits, even after a failed cell.

## Before you start (once)

**The notebook kernel** is separate from either server: any Python ≥ 3.10 with `pillow` (and
`matplotlib` for the formula image and the plot). It only sends HTTP requests.

### Option A — Linux with an L40 (`BACKEND=sglang`)

You need a **Linux x86_64 machine with an NVIDIA L40 or H100**, a **CUDA 12.x toolkit** at `CUDA_HOME`
(default `/usr/local/cuda-12.8`; kernels compile when the server starts), and **`git`** and
**[`uv`](https://docs.astral.sh/uv/)**. The model is public: no Hugging Face token.

| GPU | compute capability | status |
|---|---|---|
| **L40** (tutorial machines) | 8.9 (Ada) | the tutorial environment. `serve.sh` accepts only 9.0, so the notebook runs `serve.sh`'s other startup checks itself and skips its GPU check. The vision-encoder CUDA graphs need FA3 (Hopper only) and switch themselves off; every other `FAST=1` optimization runs |
| H100 | 9.0 (Hopper) | the GPU the serving recipe was validated on |

Build the patched SGLang server environment once, from the repo root:

```bash
SGLANG_SRC=$HOME/src/sglang-gravityocr bash inference/gravityocr/sglang/setup.sh   # prints VBIN=...
```

It fetches upstream SGLang v0.5.10 at a pinned commit, applies `sglang/gravityocr_selfspec.patch`, and
installs `sglang/requirements-lock.txt` into `$SGLANG_SRC/.venv-sglang`. Re-running only re-verifies.
Do not install Jupyter into `.venv-sglang`: `setup.sh` checks that venv against the lock and fails on
extra packages.

### Option B — Apple Silicon Mac (`BACKEND=mac`)

Install **Gravity** (macOS 14+, Apple Silicon) from
[its releases](https://github.com/trillion-labs/GravityOCR-macOS/releases): open the DMG and drag
Gravity to Applications. You do not need to open the app. `serve_mac.py` runs Gravity's bundled
Python and MLX engine on `trillionlabs/GravityOCR`, which it downloads once (2.2 GB, pinned to a
commit and checked against the Hub's SHA-256) into `~/.cache/gravityocr-mac/`.

The Mac backend is for trying the notebook, not a second serving recipe. It differs from the L40
server, so its outputs and timings are not the L40 numbers: MLX instead of SGLang, text layers
quantized to 6 bits, a 16-token draft block instead of 32, crops downscaled to a 1344 px long side.
It also imports Gravity's internal engine, written against Gravity 0.3.4; a Gravity update may break
it (`serve_mac.py` warns on other versions).

## 1. Configure and start the server

Settings come from the environment, so you can also set them before starting Jupyter. `BACKEND=auto`
picks `mac` on an Apple Silicon Mac and `sglang` elsewhere. With `sglang`, `GPU=auto` picks the
least-loaded L40 or H100, and a second one for the causal server when there is one.

In [ ]:
import os
import pathlib

BACKEND = os.environ.get("BACKEND", "auto")  # "auto" | "sglang" (Linux, L40 / H100) | "mac"
GRAVITY_APP = os.environ.get("GRAVITY_APP", "/Applications/Gravity.app")  # BACKEND=mac
SGLANG_SRC = pathlib.Path(os.environ.get("SGLANG_SRC", "~/src/sglang-gravityocr")).expanduser()
VBIN = os.environ.get("VBIN") or str(SGLANG_SRC / ".venv-sglang/bin")  # setup.sh prints it
CKPT = os.environ.get("GRAVITYOCR_MODEL", "trillionlabs/GravityOCR")   # Hub id or local directory
GPU = os.environ.get("GPU", "auto")             # "auto", or a CUDA device index such as "0"
CAUSAL_GPU = os.environ.get("CAUSAL_GPU", "auto")  # "auto": a second free GPU if there is one
COMPARE_CAUSAL = os.environ.get("COMPARE_CAUSAL", "1") == "1"  # section 5
PORT, CAUSAL_PORT = 30000, 30001
CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda-12.8")  # a CUDA 12.x toolkit (nvcc)
API_KEY = os.environ.get("API_KEY", "")  # optional; export it, never paste a key into a cell

The next cell holds the helpers (start, wait, stop, one OCR request). You can run it without reading
it; each function has a one-line description.

In [ ]:
import atexit
import json
import platform
import signal
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request

T0 = time.time()
HERE = pathlib.Path.cwd()
if not (HERE / "serve.sh").exists():
    HERE = HERE / "inference/gravityocr"
assert (HERE / "serve.sh").exists(), "start Jupyter in inference/gravityocr/ or the repo root"
if BACKEND == "auto":
    BACKEND = "mac" if sys.platform == "darwin" and platform.machine() == "arm64" else "sglang"
GRAVITY_PY = pathlib.Path(GRAVITY_APP, "Contents/Resources/runtime/bin/python")
if BACKEND == "mac":
    assert GRAVITY_PY.exists(), f"Gravity not found at {GRAVITY_APP}: see 'Before you start', option B"
else:
    assert pathlib.Path(VBIN, "python").exists(), \
        f"no server env at VBIN={VBIN}: run sglang/setup.sh first (see 'Before you start', option A)"
print(f"BACKEND = {BACKEND}")
sys.path.insert(0, str(HERE))
import ocr_request  # noqa: E402  (the CLI client in this folder; we reuse its payload builder)

servers = {}  # port -> (Popen, log path)
SUPPORTED_CC = {"8.9": "Ada (L40)", "9.0": "Hopper (H100)"}


def list_gpus():
    """{index: (name, compute capability)} of supported GPUs, least memory in use first."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.used",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True,
                             timeout=60).stdout
    except (OSError, subprocess.SubprocessError):
        return {}
    rows = [[x.strip() for x in line.split(",")] for line in out.splitlines() if line.strip()]
    return {i: (name, cap) for i, name, cap, used in sorted(rows, key=lambda r: float(r[3]))
            if cap in SUPPORTED_CC}


def check_toolchain():
    """serve.sh's startup checks other than the GPU one: a CUDA 12.x nvcc and ninja in VBIN."""
    try:
        nvcc = subprocess.run([f"{CUDA_HOME}/bin/nvcc", "--version"], capture_output=True,
                              text=True, timeout=60).stdout
    except OSError:
        nvcc = ""
    if "release 12." not in nvcc:
        raise RuntimeError(f"no CUDA 12.x nvcc at {CUDA_HOME}/bin: set CUDA_HOME")
    if not pathlib.Path(VBIN, "ninja").exists():
        raise RuntimeError(f"no ninja in VBIN={VBIN}: re-run sglang/setup.sh")


def launch(mode, gpu, port, **env):
    """Start a server (serve.sh, or serve_mac.py on a Mac) in its own process group.

    Its log goes to a private temp file: serve.sh's log records the API keys."""
    if BACKEND == "mac":
        cmd = [str(GRAVITY_PY), "-I", str(HERE / "serve_mac.py"), "--mode", mode,
               "--port", str(port), "--model", CKPT]
        where = "the Mac GPU (Gravity's MLX engine)"
    else:
        cmd = ["bash", str(HERE / "serve.sh")]
        if GPUS.get(gpu, ("", "9.0"))[1] != "9.0":
            # serve.sh accepts only compute capability 9.0: run its other checks, skip the GPU one
            check_toolchain()
            env.setdefault("SKIP_PREFLIGHT", "1")
        env = {"VBIN": VBIN, "MODE": mode, "CKPT": CKPT, "GPU": gpu, "PORT": str(port),
               "CUDA_HOME": CUDA_HOME, "CUDA_DEVICE_ORDER": "PCI_BUS_ID", **env}  # nvidia-smi order
        where = f"GPU {gpu} ({GPUS.get(gpu, ('unknown GPU',))[0]})"
    log = pathlib.Path(tempfile.gettempdir()) / f"gravityocr-{mode}-{port}.log"
    fd = os.open(log, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    proc = subprocess.Popen(cmd, stdout=fd, stderr=subprocess.STDOUT, env={**os.environ, **env},
                            start_new_session=True)
    os.close(fd)
    servers[port] = (proc, log)
    print(f"starting MODE={mode} on {where}, port {port}; log: {log}")


def stop(port):
    """Stop a server this notebook started (SIGTERM to its process group, SIGKILL after 60 s)."""
    proc, _ = servers.pop(port, (None, None))
    if proc is None or proc.poll() is not None:
        return
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped the server on port {port}")


atexit.register(lambda: [stop(p) for p in list(servers)])  # also after a failed headless run


def tail(port, n=30):
    """The last n lines of a server's log."""
    return "\n".join(servers[port][1].read_text(errors="replace").splitlines()[-n:])


def _headers():
    return {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}


def wait_until_ready(port, timeout=1800):
    """Poll /v1/models until it answers; fail with the log tail if serve.sh exits."""
    t0, last = time.time(), time.time()
    while time.time() - t0 < timeout:
        proc = servers[port][0]
        if proc.poll() is not None:
            raise RuntimeError(f"serve.sh exited with status {proc.returncode}:\n{tail(port)}")
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/models", headers=_headers())
            with urllib.request.urlopen(req, timeout=10) as r:
                model = json.loads(r.read())["data"][0]["id"]
            print(f"port {port} ready after {time.time() - t0:.0f} s (model {model!r})")
            return model
        except urllib.error.HTTPError as e:
            if e.code in (401, 403):
                raise RuntimeError("the server wants a key: export API_KEY") from None
        except (urllib.error.URLError, OSError, ValueError):
            pass
        if time.time() - last >= 60:
            print(f"  {time.time() - t0:4.0f} s  {tail(port, 1)[:140]}")
            last = time.time()
        time.sleep(3)
    raise TimeoutError(f"port {port} not ready after {timeout} s:\n{tail(port)}")


def ocr(image, prompt="Text Recognition:", port=PORT, max_tokens=4096):
    """One crop (PNG bytes) -> {"text", "tokens", "seconds"}: the payload ocr_request.py sends."""
    payload = ocr_request.build_payload(image, "image/png", prompt, "glm-ocr", max_tokens)
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", method="POST",
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json", **_headers()})
    t = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            out = json.loads(r.read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:300]}") from None
    return {"text": out["choices"][0]["message"]["content"],
            "tokens": (out.get("usage") or {}).get("completion_tokens", 0),
            "seconds": time.perf_counter() - t}

In [ ]:
GPUS = list_gpus() if BACKEND == "sglang" else {}
if GPU == "auto":
    GPU = next(iter(GPUS), "0")
if CAUSAL_GPU == "auto":
    CAUSAL_GPU = next((g for g in GPUS if g != GPU), "")  # "" = take turns on GPU

launch("self-spec", GPU, PORT)
if COMPARE_CAUSAL and BACKEND == "sglang" and CAUSAL_GPU:
    launch("causal", CAUSAL_GPU, CAUSAL_PORT)  # warms up in parallel on a second GPU

## 2. How GravityOCR works (read while the server starts)

GravityOCR is [`zai-org/GLM-OCR`](https://huggingface.co/zai-org/GLM-OCR), a vision encoder plus a 0.5B
text decoder, fine-tuned as a **block-diffusion** model: the decoder learns to fill a block of 32 masked
tokens at once, and keeps its ordinary **causal (AR) head**. **Self-speculative decoding** uses both
heads of that one checkpoint:

```
                     one round (K = 33)
 committed ──► DRAFT   [b][M][M] … [M]   last token + 32 masks, one bidirectional forward pass
 tokens               └─► proposes the next 32 tokens at once
           ──► VERIFY  one causal forward pass over the draft
                      └─► keeps the longest prefix the AR head agrees with, + the AR head's next token
           ══► up to 34 tokens per round, each one the AR head's greedy choice
```

`serve.sh` runs it on upstream SGLang v0.5.10 plus a patch (`--speculative-algorithm
SELFSPEC_DIFFUSION --speculative-num-draft-tokens 33`). `MODE=causal` serves the same weights one token
per forward pass: the reference. The first start on a machine compiles FlashInfer and Triton kernels
and captures CUDA graphs; later starts are faster. On a Mac, `serve_mac.py` runs the same two modes
with Gravity's MLX engine, drafting blocks of 16 tokens.

In [ ]:
wait_until_ready(PORT)
for line in servers[PORT][1].read_text(errors="replace").splitlines():
    # the mask id must come from the checkpoint's block_diffusion.json; on L40 the ViT graphs are OFF
    if any(tag in line for tag in ("[selfspec-diffusion]", "[ss-vit-graph]", "[serve_mac] mode=")):
        print(line.strip()[:200])

if COMPARE_CAUSAL and BACKEND == "mac":
    launch("causal", GPU, CAUSAL_PORT)  # the checkpoint is cached now, so this loads in seconds

## 3. First request

The server speaks the OpenAI chat-completions API. One request is **one cropped region** plus a task
prompt, the image first, as a `data:` URL, with `temperature: 0`:

```json
{"model": "glm-ocr", "max_tokens": 4096, "temperature": 0,
 "messages": [{"role": "user", "content": [
   {"type": "image_url", "image_url": {"url": "data:image/png;base64,iVBORw0KGgo..."}},
   {"type": "text", "text": "Text Recognition:"}]}]}
```

The first check needs no sample image: render text into a PNG and read it back.

In [ ]:
from IPython.display import HTML, Image, Math, display

hello = ocr_request.render_text_png("Hello GravityOCR")
display(Image(hello))
r = ocr(hello)
print(repr(r["text"]), f"({r['tokens']} tokens, {r['seconds']:.2f} s)")
print("PASS: read back" if " ".join(r["text"].split()) == "Hello GravityOCR" else "CHECK: text differs")

## 4. Text, a table and a formula

| prompt | crop | output |
|---|---|---|
| `Text Recognition:` | a text block | plain text |
| `Table Recognition:` | one table | HTML |
| `Formula Recognition:` | one display formula | LaTeX |

The crops below are clean renders made in this kernel. Real scans are harder: add your own in
`MY_CROPS`. Crop **one region per request**; a full page needs a layout detector in front.

In [ ]:
import io

from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont


def table_png(rows, w=170, h=46):
    """A ruled table with a grey header row."""
    img = PILImage.new("RGB", (w * len(rows[0]) + 2, h * len(rows) + 2), "white")
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default(size=22)
    except TypeError:  # Pillow < 10.1
        font = ImageFont.load_default()
    for i, row in enumerate(rows):
        for j, cell in enumerate(row):
            box = (j * w, i * h, (j + 1) * w, (i + 1) * h)
            d.rectangle(box, outline="black", width=2, fill="#e8e8e8" if i == 0 else "white")
            d.text((box[0] + 12, box[1] + 11), cell, fill="black", font=font)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def formula_png(latex):
    """A display formula rendered by matplotlib mathtext (plain text without matplotlib)."""
    try:
        from matplotlib.figure import Figure  # not pyplot: leaves the notebook's backend alone
    except ImportError:
        return ocr_request.render_text_png(latex)
    fig = Figure(figsize=(0.01, 0.01))
    fig.text(0, 0, f"${latex}$", fontsize=30)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight", pad_inches=0.25, facecolor="white")
    return buf.getvalue()


CROPS = {
    "paragraph": (ocr_request.render_text_png(
        "GravityOCR drafts a block of tokens with its\ndiffusion head and verifies them with the\n"
        "causal head of the same checkpoint.", font_size=28), "Text Recognition:"),
    "table": (table_png([["Model", "Mode", "Pass"], ["GravityOCR", "self-spec", "draft"],
                         ["GravityOCR", "causal", "verify"]]), "Table Recognition:"),
    "formula": (formula_png(r"\int_0^1 x^2\,dx = \frac{1}{3}"), "Formula Recognition:"),
}
MY_CROPS = {}  # {"name": ("path/to/crop.png", one of the three prompts)}
for name, (path, prompt) in MY_CROPS.items():
    CROPS[name] = (pathlib.Path(path).expanduser().read_bytes(), prompt)

for name, (img, prompt) in CROPS.items():
    print(f"── {name}  ({prompt})")
    display(Image(img))
    r = ocr(img, prompt)
    print(r["text"], f"\n[{r['tokens']} tokens, {r['seconds']:.2f} s]")
    if prompt.startswith("Table"):
        display(HTML(r["text"]))
    elif prompt.startswith("Formula"):
        display(Math(r["text"].strip().strip("$").strip()))

The same request from a shell, with the CLI client in this folder (standard library only; `--synthetic`
needs Pillow). It reads `API_KEY` from the environment.

In [ ]:
!"{sys.executable}" "{HERE}/ocr_request.py" --synthetic "Hello GravityOCR"

## 5. Self-spec vs causal

Both modes decode the same weights. Expect **mostly identical text**: under bf16 the draft and verify
passes can resolve a near-tie between two tokens differently from causal decoding, so a crop can
diverge from that token on (the PR's GPU check: 92 of 100 crops identical). Expect self-spec to gain
most on **long outputs**, since a round commits up to 34 tokens. These are single requests, one at a
time, after one warm-up request per server: latency, not throughput. Measure on your machine; the
recipe makes no speed claim. With `BACKEND=mac` the draft block is 16 tokens (up to 17 per round) and
the backend differs, so Mac numbers say nothing about the L40.

In [ ]:
BENCH = dict(CROPS)
BENCH["long paragraph"] = (ocr_request.render_text_png("\n".join(
    f"{i}. Self-speculative decoding commits up to thirty-four tokens per round."
    for i in range(1, 13)), font_size=24), "Text Recognition:")


def run_set(port):
    ocr(hello, port=port)  # warm-up
    return {name: ocr(img, prompt, port=port) for name, (img, prompt) in BENCH.items()}


res = {"self-spec": run_set(PORT)}
if COMPARE_CAUSAL:
    if CAUSAL_PORT not in servers:  # one GPU: stop self-spec, then start causal on the same GPU
        stop(PORT)
        launch("causal", GPU, CAUSAL_PORT)
    wait_until_ready(CAUSAL_PORT)
    res["causal"] = run_set(CAUSAL_PORT)

print(f"{'crop':<15}{'tokens':>7}{'self-spec s':>13}{'causal s':>10}{'speedup':>9}  same text")
for name in BENCH:
    s, c = res["self-spec"][name], res.get("causal", {}).get(name)
    print(f"{name:<15}{s['tokens']:>7}{s['seconds']:>13.3f}" + (
        f"{c['seconds']:>10.3f}{c['seconds'] / s['seconds']:>8.2f}x  "
        f"{'yes' if c['text'] == s['text'] else 'NO'}" if c else ""))
if "causal" in res:
    same = sum(res["causal"][n]["text"] == res["self-spec"][n]["text"] for n in BENCH)
    print(f"\nPASS: compared {len(BENCH)} crops, {same} identical")

In [ ]:
if "causal" in res:
    try:
        from matplotlib.figure import Figure
    except ImportError:
        Figure = None
    if Figure:
        names = list(BENCH)
        fig = Figure(figsize=(7, 2.6))
        ax = fig.subplots()
        x = range(len(names))
        ax.bar([i - 0.2 for i in x], [res["causal"][n]["seconds"] for n in names], 0.4,
               label="causal", color="#9AA5B1")
        ax.bar([i + 0.2 for i in x], [res["self-spec"][n]["seconds"] for n in names], 0.4,
               label="self-spec", color="#3D6FB6")
        ax.set_xticks(list(x), names)
        ax.set_ylabel("seconds per crop")
        ax.legend(frameon=False)
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=100)  # a PNG shows in any Jupyter and in headless runs
        display(Image(buf.getvalue()))

## 6. Clean up

Stops the servers this notebook started (they are also stopped when the kernel exits).

In [ ]:
for port in list(servers):
    stop(port)
print(f"tutorial time: {(time.time() - T0) / 60:.1f} min (first cell to here, server starts included)")

**Next:** your own crops in `MY_CROPS`; more replicas with `DP=2 GPU=0,1 bash serve.sh`; a fine-tune
with `CKPT=/path/to/ckpt` once it passes the appendix's checkpoint check; the design notes in
[`trillion-labs/GravityOCR`](https://github.com/trillion-labs/GravityOCR).

---

# Appendix (not part of the 10 minutes)

## A. What the L40 recipe contains

The repo carries a **patch against a pinned upstream commit plus four new files**, not a fork; the
cell below lists them.

| new file (`python/sglang/srt/…`) | role |
|---|---|
| `speculative/selfspec_diffusion_worker.py` | the `SELFSPEC_DIFFUSION` worker: one bidirectional DRAFT pass, one causal VERIFY pass, accept the longest matching prefix |
| `speculative/selfspec_fast_attn.py` | `SELFSPEC_FASTATTN`: draft/verify attention with FlashInfer's built-in mask modes, no device→host copy |
| `speculative/selfspec_fast_commit.py` | `SELFSPEC_FAST_COMMIT`: a round's commit bookkeeping in one Triton kernel, one readback |
| `multimodal/glm_vit_graph_runner.py` | `SS_GLM_VIT_GRAPH_MAXM`: the vision encoder as CUDA graphs, one per patch count. Needs SGLang's FA3 vision attention (Hopper); on L40 it logs `OFF` and the encoder runs eagerly |

`serve.sh` turns the fast paths on with `FAST=1` (default); `FAST=0` serves the unoptimized runtime.

In [ ]:
import re

sg = HERE / "sglang"
patch = (sg / "gravityocr_selfspec.patch").read_text()
touched = re.findall(r"^diff --git a/(\S+) b/", patch, flags=re.M)
lock = dict(re.findall(r"^([A-Za-z0-9._-]+)==(\S+)", (sg / "requirements-lock.txt").read_text(),
                       flags=re.M))
print(f"upstream sgl-project/sglang @ {(sg / 'BASE_COMMIT.txt').read_text()[:12]} (v0.5.10)")
print(f"patch: {len(touched)} files —", ", ".join(f.split("/")[-1] for f in touched))
for p in sorted((sg / "python").rglob("*.py")):
    print(f"new:   {p.relative_to(sg / 'python')} ({len(p.read_text().splitlines())} lines)")
print(f"lock:  {len(lock)} pins, among them", ", ".join(
    f"{k} {lock[k]}" for k in ("torch", "transformers", "flashinfer-python") if k in lock))

## B. Check a checkpoint

What a checkpoint needs to be servable here ([`MODEL_CONTRACT.md`](MODEL_CONTRACT.md)). The key file
for self-spec is `block_diffusion.json`: with a wrong `mask_id` the server still answers correctly but
almost no draft is accepted, so self-spec is no faster. Reads small files only (from `CKPT`, a local
directory or the Hub).

In [ ]:
def ckpt_json(name):
    if pathlib.Path(CKPT).expanduser().is_dir():
        return json.loads((pathlib.Path(CKPT).expanduser() / name).read_text())
    url = f"https://huggingface.co/{CKPT}/resolve/main/{name}"
    with urllib.request.urlopen(url, timeout=120) as r:
        return json.loads(r.read())


cfg, bd, tok = (ckpt_json(f) for f in ("config.json", "block_diffusion.json", "tokenizer.json"))
mask = {t["id"]: t["content"] for t in tok.get("added_tokens", [])}.get(bd.get("mask_id"))
arch, ar_w, bd_size = cfg.get("architectures"), bd.get("ar_loss_weight") or 0, bd.get("bd_size", 0)
for ok, what in [
    (arch == ["GlmOcrForConditionalGeneration"], f"architecture {arch}"),
    ("auto_map" not in cfg, "no auto_map (no remote code)"),
    (ar_w > 0, f"ar_loss_weight {ar_w} > 0 (causal head trained)"),
    (mask == "<|mask|>", f"mask_id {bd.get('mask_id')} is {mask!r} in tokenizer.json"),
    (bd_size + 1 == 33, f"bd_size {bd_size} -> DRAFT {bd_size + 1} (serve.sh default 33)"),
]:
    print("✓" if ok else "✗", what)

## C. Security

The defaults suit a single-user machine; read [README → Security](README.md#security) before sharing
the server.

- **Loopback by default.** Another `SERVE_HOST` requires `API_KEY`, which `serve.sh` takes from the
  environment only and never puts on a command line.
- **Never expose the port directly.** SGLang fetches `http(s)://` image URLs and reads `file://` paths
  on the server. Put a proxy in front that forwards only `/v1/chat/completions` and `/v1/models` with
  `data:` image URLs.
- **Management endpoints** (weight reloads, profiling) are locked with a random admin key whenever the
  server runs one tokenizer worker. With the `FAST=1` default of two workers and no key they are open to
  every user on the host: on a shared machine set `API_KEY` or `TOKENIZER_WORKERS=1`.
- **The server log records the keys**: keep it private, or `EXTRA_ARGS="--log-level warning"`.

## D. Troubleshooting

| symptom | fix |
|---|---|
| `no server env at VBIN=…` | run `sglang/setup.sh` (Before you start), or export `VBIN` from its last line |
| `serve.sh exited with status 2` | a setting was refused; the log tail names the variable |
| `GPU … has compute capability …` | `GPU` points at a GPU that is neither an L40 (8.9) nor an H100 (9.0); `GPU=auto` picks one |
| `no CUDA 12.x nvcc` / CUDA 13.x | point `CUDA_HOME` at a CUDA 12.x toolkit |
| `setup.sh`: `not in the lock: …` | something was installed into `.venv-sglang` by hand: remove it |
| HTTP 400 *exceeds the model's maximum context length* | keep `max_tokens` ≤ 4096 (`CONTEXT` is 16384) |
| FlashInfer workspace overflow at CUDA-graph capture | lower `MAXREQ` or `GRAPHBS` |
| self-spec no faster than causal | the `[selfspec-diffusion] … mask_id=…` line in section 2; run appendix B |
| Mac: `Gravity not found at …` | install Gravity (Before you start, option B), or set `GRAVITY_APP` |
| Mac: the first start takes minutes | `serve_mac.py` is downloading the 2.2 GB checkpoint once; `tail(PORT)` shows it |
| Mac: `warning: Gravity X.Y.Z` in the log | an untested Gravity version; if the server fails, use Gravity 0.3.4 |
| a server died | `print(tail(PORT, 200))` |